# 02. Parse GTFS-RT archive into DuckDB

Reads raw protobuf snapshots from `data/raw/gtfs_rt/`, parses each `trip_update` message into rows, and writes to a `gtfsrt` table in the project DuckDB. Also loads the GTFS static tables (`trips`, `calendar`, `calendar_dates`, `stop_times`).

This notebook is incremental: it skips service dates already loaded in `gtfsrt`, so it's safe to rerun as the archive grows.

**Prerequisites:** GTFS static feed in `data/google_bus/`, raw `.pb` files in `data/raw/gtfs_rt/`.

**Outputs:** `gtfsrt`, `trips`, `calendar`, `calendar_dates`, `stop_times` in `data/transit_kpi.duckdb`.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone, timedelta
import pytz
from google.transit import gtfs_realtime_pb2
from google.protobuf.message import DecodeError

PROJECT_ROOT = Path(r'C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework')
DB_PATH      = PROJECT_ROOT / 'data' / 'transit_kpi.duckdb'
ARCHIVE_DIR  = PROJECT_ROOT / 'data' / 'raw' / 'gtfs_rt'
STATIC_DIR   = PROJECT_ROOT / 'data' / 'google_bus'

EASTERN = pytz.timezone('America/New_York')

con = duckdb.connect(str(DB_PATH))

print(f'Project root:  {PROJECT_ROOT}')
print(f'Database:      {DB_PATH}')
print(f'Archive dir:   {ARCHIVE_DIR}')
print()
print('Existing tables:')
print(con.sql('SHOW TABLES').df().to_string(index=False))

## Service date assignment

GTFS uses a rolling service day, not a calendar day. Some bus trips that start at 11:55 PM finish at 12:14 AM and are scheduled in static GTFS as 24:xx times on the *previous* service date. If we assign post-midnight snapshots to the next calendar day, the matching breaks.

The function below uses a 4 AM Eastern boundary. Snapshots taken between midnight and 4 AM stay on the previous service date. Route 23's last Owl trip ends at 3:54 AM and the first day-service trip starts at 4:10 AM, so 4 AM falls in a clean gap.

DST is handled by `pytz.localize` rather than fixed UTC offsets.

In [ ]:
def snapshot_to_service_date(snapshot_ts_utc: datetime) -> tuple[str, int]:
    """Convert a UTC snapshot timestamp to (service_date_iso, midnight_unix)."""
    snapshot_eastern = snapshot_ts_utc.astimezone(EASTERN)

    if snapshot_eastern.hour < 4:
        service_date = snapshot_eastern.date() - timedelta(days=1)
    else:
        service_date = snapshot_eastern.date()

    midnight = EASTERN.localize(
        datetime(service_date.year, service_date.month, service_date.day, 0, 0, 0)
    )
    return service_date.isoformat(), int(midnight.timestamp())


# Sanity checks for the three boundary cases
test_daytime = datetime(2026, 4, 7, 14, 30, 0, tzinfo=timezone.utc)
assert snapshot_to_service_date(test_daytime) == ('2026-04-07', 1775534400)

test_post_midnight = datetime(2026, 4, 8, 5, 30, 0, tzinfo=timezone.utc)  # 1:30 AM EDT
assert snapshot_to_service_date(test_post_midnight) == ('2026-04-07', 1775534400)

test_after_4am = datetime(2026, 4, 8, 8, 30, 0, tzinfo=timezone.utc)  # 4:30 AM EDT
assert snapshot_to_service_date(test_after_4am) == ('2026-04-08', 1775620800)

print('Service-date conversions:')
print(f'  10:30 AM EDT April 7  -> {snapshot_to_service_date(test_daytime)}')
print(f'  1:30 AM EDT April 8   -> {snapshot_to_service_date(test_post_midnight)}')
print(f'  4:30 AM EDT April 8   -> {snapshot_to_service_date(test_after_4am)}')

## Parse the protobuf archive

The `gtfsrt` table is created with an explicit schema and appended to one service date at a time. Days already present are skipped, so re-running this notebook only processes new days from the archiver.

In [ ]:
con.execute("""
    CREATE TABLE IF NOT EXISTS gtfsrt (
        snapshot_ts            VARCHAR,
        service_date           VARCHAR,
        midnight_unix          BIGINT,
        trip_id                VARCHAR,
        route_id               VARCHAR,
        schedule_relationship  INTEGER,
        stop_sequence          INTEGER,
        stop_id                VARCHAR,
        predicted_unix         BIGINT,
        predicted_ts           VARCHAR
    )
""")

existing_dates = set(
    con.sql('SELECT DISTINCT service_date FROM gtfsrt').df()['service_date'].tolist()
)
print(f'Service dates already loaded: {len(existing_dates)}')
if existing_dates:
    print(f'  range: {min(existing_dates)} to {max(existing_dates)}')

In [ ]:
# Group raw files by service date so we can process and insert one day at a time
pb_files = sorted(ARCHIVE_DIR.glob('*.pb'))
print(f'Total .pb files in archive: {len(pb_files)}')

files_by_date: dict[str, list[Path]] = {}
skipped_unparseable_name = 0

for pb_file in pb_files:
    try:
        snapshot_ts_utc = datetime.strptime(
            pb_file.stem, '%Y%m%dT%H%M%SZ'
        ).replace(tzinfo=timezone.utc)
    except ValueError:
        skipped_unparseable_name += 1
        continue

    service_date_str, _ = snapshot_to_service_date(snapshot_ts_utc)
    files_by_date.setdefault(service_date_str, []).append(pb_file)

print(f'Skipped (unparseable filename): {skipped_unparseable_name}')
print(f'Distinct service dates:         {len(files_by_date)}')
if files_by_date:
    print(f'  range: {min(files_by_date)} to {max(files_by_date)}')
    print()
    print('Files per service date:')
    for d in sorted(files_by_date):
        print(f'  {d}: {len(files_by_date[d]):4d} files')

In [ ]:
# Parse one day at a time and insert into DuckDB. The day-at-a-time pattern
# keeps memory bounded; a full archive could blow up RAM if held all at once.

dates_to_process = sorted(d for d in files_by_date if d not in existing_dates)
print(f'Dates to process: {len(dates_to_process)}')
print(f'Dates skipped (already loaded): {len(files_by_date) - len(dates_to_process)}')
print()

failed_files: list[tuple[str, str]] = []
total_records_inserted = 0

for service_date_str in dates_to_process:
    day_files = files_by_date[service_date_str]
    records = []

    for pb_file in day_files:
        try:
            snapshot_ts_utc = datetime.strptime(
                pb_file.stem, '%Y%m%dT%H%M%SZ'
            ).replace(tzinfo=timezone.utc)
            _, midnight_unix = snapshot_to_service_date(snapshot_ts_utc)

            feed = gtfs_realtime_pb2.FeedMessage()
            with open(pb_file, 'rb') as f:
                feed.ParseFromString(f.read())

        except (DecodeError, OSError, ValueError) as e:
            failed_files.append((pb_file.name, f'{type(e).__name__}: {e}'))
            continue

        for entity in feed.entity:
            if not entity.HasField('trip_update'):
                continue

            tu = entity.trip_update
            trip_id   = tu.trip.trip_id
            route_id  = tu.trip.route_id
            sched_rel = tu.trip.schedule_relationship

            for stu in tu.stop_time_update:
                # Some predictions only carry one of arrival/departure; take whichever
                if stu.HasField('arrival'):
                    predicted_time = stu.arrival.time
                elif stu.HasField('departure'):
                    predicted_time = stu.departure.time
                else:
                    continue

                records.append({
                    'snapshot_ts':           snapshot_ts_utc.isoformat(),
                    'service_date':          service_date_str,
                    'midnight_unix':         midnight_unix,
                    'trip_id':               trip_id,
                    'route_id':              route_id,
                    'schedule_relationship': sched_rel,
                    'stop_sequence':         stu.stop_sequence,
                    'stop_id':               stu.stop_id,
                    'predicted_unix':        predicted_time,
                    'predicted_ts':          datetime.fromtimestamp(
                                                 predicted_time, tz=timezone.utc
                                             ).isoformat(),
                })

    if records:
        df = pd.DataFrame(records)
        con.execute('INSERT INTO gtfsrt SELECT * FROM df')
        total_records_inserted += len(records)
        print(f'  {service_date_str}: {len(day_files):4d} files, {len(records):>9,} records')
        del df, records
    else:
        print(f'  {service_date_str}: {len(day_files):4d} files, 0 records (all failed?)')

print()
print(f'Total records inserted: {total_records_inserted:,}')
print(f'Failed files:           {len(failed_files)}')
if failed_files:
    print('First 10 failures:')
    for name, err in failed_files[:10]:
        print(f'  {name}: {err}')

## Load the GTFS static tables

These get recreated from disk each run. Quick operation, ensures the schedule tables match whatever's in `data/google_bus/`. The `stop_times` table gets two extra columns: arrival and departure times converted from `HH:MM:SS` strings to seconds-past-midnight integers, which makes downstream filtering much faster.

In [ ]:
con.execute('DROP TABLE IF EXISTS trips')
con.execute(f"""
    CREATE TABLE trips AS
    SELECT * FROM read_csv_auto('{STATIC_DIR / 'trips.txt'}')
""")

con.execute('DROP TABLE IF EXISTS calendar')
con.execute(f"""
    CREATE TABLE calendar AS
    SELECT * FROM read_csv_auto('{STATIC_DIR / 'calendar.txt'}')
""")

con.execute('DROP TABLE IF EXISTS calendar_dates')
con.execute(f"""
    CREATE TABLE calendar_dates AS
    SELECT * FROM read_csv_auto('{STATIC_DIR / 'calendar_dates.txt'}')
""")

con.execute('DROP TABLE IF EXISTS stop_times')
con.execute(f"""
    CREATE TABLE stop_times AS
    SELECT
        trip_id,
        stop_sequence,
        stop_id,
        arrival_time,
        departure_time,
        timepoint,
        CAST(SPLIT_PART(arrival_time, ':', 1) AS INTEGER) * 3600 +
        CAST(SPLIT_PART(arrival_time, ':', 2) AS INTEGER) * 60 +
        CAST(SPLIT_PART(arrival_time, ':', 3) AS INTEGER) AS arrival_seconds,
        CAST(SPLIT_PART(departure_time, ':', 1) AS INTEGER) * 3600 +
        CAST(SPLIT_PART(departure_time, ':', 2) AS INTEGER) * 60 +
        CAST(SPLIT_PART(departure_time, ':', 3) AS INTEGER) AS departure_seconds
    FROM read_csv_auto('{STATIC_DIR / 'stop_times.txt'}')
""")

print('Static GTFS tables loaded:')
for tbl in ['trips', 'calendar', 'calendar_dates', 'stop_times']:
    n = con.sql(f'SELECT COUNT(*) AS n FROM {tbl}').fetchone()[0]
    print(f'  {tbl:18s} {n:>10,} rows')

## Summary

In [ ]:
print('=== gtfsrt table summary ===')
print()

print('Row count and date range:')
print(con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT service_date) AS distinct_dates,
        MIN(service_date) AS first_date,
        MAX(service_date) AS last_date
    FROM gtfsrt
""").df().to_string(index=False))
print()

print('Records per service_date:')
print(con.sql("""
    SELECT service_date, COUNT(*) AS records
    FROM gtfsrt
    GROUP BY service_date
    ORDER BY service_date
""").df().to_string(index=False))
print()

print('Schedule_relationship distribution (0 = SCHEDULED):')
print(con.sql("""
    SELECT
        schedule_relationship,
        COUNT(*) AS n,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM gtfsrt
    GROUP BY schedule_relationship
    ORDER BY schedule_relationship
""").df().to_string(index=False))
print()

print('Top 10 routes by record count (focus routes 23 and 47):')
print(con.sql("""
    SELECT route_id, COUNT(*) AS records
    FROM gtfsrt
    GROUP BY route_id
    ORDER BY records DESC
    LIMIT 10
""").df().to_string(index=False))